In [4]:
# %% [markdown]
# # 14 — Sparse KAN v3: Does re-searching hyperparameters recover v1→v2 losses?
#
# ═══════════════════════════════════════════════════════════════════════
# WHY THIS NOTEBOOK EXISTS
# ═══════════════════════════════════════════════════════════════════════
#
# v2 (fixed init + eps=1e-5) reused v1's stored hyperparameters rather
# than re-searching -- a deliberate methodological choice at the time,
# to isolate "does the fix help" from "does more tuning help". Several
# configs showed large validation drops from v1 to v2 at those SAME
# hyperparameters. This notebook tests a specific, narrower question:
# were v1's hyperparameters simply mismatched to the corrected model
# (tuned to compensate for the old bugs), or does the fix genuinely not
# help regardless of tuning?
#
# CAVEAT, stated up front and worth repeating in the results section:
# four of these five configs were selected because they showed the
# LARGEST v1->v2 drops. Extreme values partially regress toward the mean
# on any re-measurement from noise alone, independent of any real causal
# effect. A "recovery" here is suggestive, not proof that hyperparameter
# mismatch was the whole story -- especially given validation AUC/R2 on
# these splits is itself noisy (small crash-event counts; seed-to-seed
# AUC std as high as 0.13 was measured in the original 3-seed sweep).
#
# FIVE PROBE CONFIGS: four selected on largest v1->v2 drop (spanning both
# datasets, both target types, both L1 branches), plus one CONTROL config
# whose v1->v2 delta was near zero -- included specifically so a "no
# recovery" result on the control isn't misread as evidence against the
# other four (the control's v1 params should already be ~optimal for the
# fixed model, so enqueue_trial(old_best) is expected to roughly hold,
# not need genuine improvement).
#
# ┌───┬──────────────────────────────────────────┬────────┬────────┬────────┬───────┐
# │ # │ Config                                    │ v1 val │ v2 val │  delta │  L1?  │
# ├───┼──────────────────────────────────────────┼────────┼────────┼────────┼───────┤
# │ 1 │ agg_full_moments/Split_B/binary/seed42     │ 0.7010 │ 0.5232 │ -0.178 │  Yes  │
# │ 2 │ agg_means/Split_B/binary/seed42            │ 0.8125 │ 0.5888 │ -0.224 │  Yes  │
# │ 3 │ agg_full_moments/Split_C/continuous/seed42 │ 0.2830 │ 0.0600 │ -0.223 │  Yes  │
# │ 4 │ agg_means/Split_C/continuous/seed42        │ 0.2832 │ 0.0011 │ -0.282 │  Yes  │
# │ 5 │ agg_means/Split_C/binary/seed42 (CONTROL)  │ 0.7593 │ 0.7453 │ -0.014 │  n/a  │
# └───┴──────────────────────────────────────────┴────────┴────────┴────────┴───────┘
#
# Note on #4: seed 42's v2 val (0.0011) is used here rather than the
# seed-456 figure (0.1043) originally quoted, since seed 42 shows the
# larger drop and this notebook fixes seed=42 throughout for all probes.
#
# METHOD: 15 Optuna trials per config, WITH-L1 SEARCH SPACE ONLY (all
# four non-control probes used L1 in v1; the control doesn't need L1
# specified since its old params are enqueued and it's not meant to
# improve). study.enqueue_trial(old_best_params) is inserted as trial 0
# for every config -- this guarantees v3's best value is >= v2's by
# construction (the old point is always in the search space as a
# candidate), so any improvement seen is attributable to the NEW trials,
# not to re-running the old config and getting lucky.
#
# DECISION RULE (tightened from the original "within 0.03"):
#   - "Recovered": v3 val is within 0.03 of the ORIGINAL v1 val. 0.03 is
#     deliberately tighter than the measured seed-to-seed AUC noise floor
#     (std up to 0.13 in the original sweep), so a "recovered" verdict is
#     a real signal, not noise.
#   - "Not recovered": v3 val stays at or near v2 levels. Given the
#     tightness of the 0.03 bar relative to measured noise, this is also
#     informative on its own -- it does NOT necessarily mean nothing
#     changed, but it does mean re-searching did not close the gap.
#   - Read the CONTROL alongside the four: the control is expected to
#     roughly hold at its (already near-v1) value, since there's little
#     room for genuine improvement there. If the control ALSO moves
#     substantially, that's a sign the whole exercise is more noise-driven
#     than a real hyperparameter-mismatch story.
#   - Final call: if >=3 of the 4 non-control probes recover to within
#     0.03 of v1, a full re-search sweep is likely worth the overnight
#     run. If fewer than that recover, treat the fix as validated
#     numerically-but-not-predictively for these configs, and report that
#     directly -- a legitimate, honest finding either way.
#
# ═══════════════════════════════════════════════════════════════════════
# RESULTS ISOLATION
# ═══════════════════════════════════════════════════════════════════════
#
# Writes to a fresh directory (.../sparse_kan_v3_probe/), separate from
# BOTH the original sparse_kan/ and the retrain-only sparse_kan_v2/.
# Reads from sparse_kan/'s Optuna .db files (read-only, for the
# enqueue_trial starting point) and nothing else. No existing results are
# touched or overwritten anywhere.
#
# Estimated runtime: 5 configs x 15 trials x ~10-20s/trial ~= 15-25 min.

# %%
# ── COLAB SETUP ──
!pip install -q git+https://github.com/Blealtan/efficient-kan.git optuna

from google.colab import drive
drive.mount("/content/drive")

import sys
sys.path.insert(0, "/content/drive/MyDrive/Thesis/Code")

# %%
import json
import numpy as np
import pandas as pd
import random
import time
import torch
import optuna
from pathlib import Path

from data_utils import load_split, get_dataloaders, get_device, load_theme_assignment
from training import train_model, save_checkpoint
from evaluation import evaluate_model, save_predictions, compute_calibration
from sparse_kan import SparseKAN, sparse_kan_edge_l1

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# ═══════════════════════════════════════════════════════════════════════════
# SANITY CHECK — confirm the live sparse_kan.py has both fixes before
# spending any GPU time. Identical to the check used in sparse_kan_v2.
# ═══════════════════════════════════════════════════════════════════════════

print("=" * 90)
print("SANITY CHECK: confirming init fix + eps=1e-5/affine=False")
print("=" * 90)

_fake_tax = pd.DataFrame({
    "column":        ["f1", "f2", "f3", "f4", "f5"],
    "subtheme_id":   ["01_01"]*5,
    "subtheme_name": ["SubA"]*5,
    "theme_id":      ["01"]*5,
    "theme_name":    ["ThemeX"]*5,
})
_fcols = ["f1", "f2", "f3", "f4", "f5"]
N_DUMMY = len(_fcols)

_m = SparseKAN.from_taxonomy(_fake_tax, _fcols, grid_size=5, spline_order=3,
                             grid_range=[-1, 1])

assert abs(_m.bn1.eps - 1e-5) < 1e-9, f"bn1.eps={_m.bn1.eps}, expected 1e-5. STOP."
assert abs(_m.bn2.eps - 1e-5) < 1e-9, f"bn2.eps={_m.bn2.eps}, expected 1e-5. STOP."
assert _m.bn1.affine is False, f"bn1.affine={_m.bn1.affine}, expected False. STOP."
assert _m.bn2.affine is False, f"bn2.affine={_m.bn2.affine}, expected False. STOP."
print(f"  ✓ eps=1e-5, affine=False on both bn1/bn2")

_active = _m.layer0.mask[0].bool()
_fan_in = int(_active.sum().item())
_bound = 1.0 / _fan_in**0.5
for _pname, _W in [("base_weight",   _m.layer0.base_weight.data),
                   ("spline_scaler", _m.layer0.spline_scaler.data)]:
    _ratio = _W[0][_active].abs().max().item() / _bound
    assert _ratio > 0.5, f"{_pname} ratio={_ratio:.3f} -- STALE FILE. STOP."
    print(f"  ✓ init fix on {_pname}: ratio = {_ratio:.3f}")

assert _m.verify_masking(), "masking broken at construction. STOP."
_m.train()
assert _m(torch.randn(16, N_DUMMY)).shape == (16, 1)
print("  ✓ forward pass OK")

del _fake_tax, _fcols, _m, _active, _fan_in, _bound, N_DUMMY

print("\n" + "=" * 90)
print("SANITY CHECK PASSED")
print("=" * 90)


# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════

SPLITS_DIR  = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/04_splits")
THEMES_DIR  = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/05_themes")
HUBER_DELTA_PATH = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/03_targets/huber_delta.json")

# Read v1's original Optuna .db files (read-only, for enqueue_trial only)
V1_RESULTS_DIR = Path("/content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/sparse_kan")

# Write to a brand new directory -- touches neither sparse_kan/ nor sparse_kan_v2/
RESULTS_DIR = Path("/content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/sparse_kan_v3_probe")

GRID_SIZE    = 14
SPLINE_ORDER = 3
GRID_RANGE   = [-5.5, 5.5]

SEED = 42  # every probe uses seed 42
N_TRIALS = 15

with open(HUBER_DELTA_PATH) as f:
    HUBER_DELTAS = json.load(f)["deltas"]

def get_huber_delta(split_name):
    return HUBER_DELTAS[f"{split_name}/market"]


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ═══════════════════════════════════════════════════════════════════════════
# THE FIVE PROBE CONFIGS
# ═══════════════════════════════════════════════════════════════════════════

PROBE_CONFIGS = [
    {"label": "1_afm_SplitB_binary",     "dataset": "agg_full_moments", "split": "Split_B",
     "target": "binary",     "v1_val": 0.7010, "v2_val": 0.5232, "use_l1_search": True},
    {"label": "2_means_SplitB_binary",   "dataset": "agg_means",        "split": "Split_B",
     "target": "binary",     "v1_val": 0.8125, "v2_val": 0.5888, "use_l1_search": True},
    {"label": "3_afm_SplitC_continuous", "dataset": "agg_full_moments", "split": "Split_C",
     "target": "continuous", "v1_val": 0.2830, "v2_val": 0.0600, "use_l1_search": True},
    {"label": "4_means_SplitC_continuous","dataset": "agg_means",       "split": "Split_C",
     "target": "continuous", "v1_val": 0.2832, "v2_val": 0.0011, "use_l1_search": True},
    {"label": "5_CONTROL_means_SplitC_binary", "dataset": "agg_means",  "split": "Split_C",
     "target": "binary",     "v1_val": 0.7593, "v2_val": 0.7453, "use_l1_search": True,
     "is_control": True},
]


# ═══════════════════════════════════════════════════════════════════════════
# LOAD TAXONOMIES
# ═══════════════════════════════════════════════════════════════════════════

taxonomy_dfs = {}
for ds in {c["dataset"] for c in PROBE_CONFIGS}:
    taxonomy_dfs[ds] = load_theme_assignment(ds, THEMES_DIR)

device = get_device()


# ═══════════════════════════════════════════════════════════════════════════
# LOAD v1's BEST PARAMS — used as the enqueue_trial(old_best) starting point
# ═══════════════════════════════════════════════════════════════════════════

def load_v1_best_params(model_name, target_type, split_name, seed=42):
    """
    Read-only load of v1's original with-L1 study for this config, to use
    as the enqueue_trial() starting point. Does NOT read or write anywhere
    else in v1's results directory.
    """
    study_dir = V1_RESULTS_DIR / f"seed_{seed}" / "optuna"
    path_l1 = study_dir / f"{model_name}_{target_type}_{split_name}_with_L1_seed{seed}.db"

    if not path_l1.exists():
        raise FileNotFoundError(f"v1 with-L1 study not found at {path_l1}")

    study = optuna.load_study(
        study_name=f"{model_name}_{target_type}_{split_name}_with_L1_seed{seed}",
        storage=f"sqlite:///{path_l1}",
    )
    return study.best_params, study.best_value


# ═══════════════════════════════════════════════════════════════════════════
# MODEL FACTORY — with-L1 search space only (matches v1's with-L1 phase)
# ═══════════════════════════════════════════════════════════════════════════

def make_model_factory_with_l1(feature_cols, taxonomy_df, huber_delta, target_type):
    def factory(trial):
        lr           = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
        batch_size   = trial.suggest_categorical("batch_size", [64, 128, 256])
        reg_weight   = trial.suggest_float("reg_weight", 1e-7, 1e-3, log=True)

        model = SparseKAN.from_taxonomy(
            taxonomy_df, feature_cols,
            grid_size=GRID_SIZE, spline_order=SPLINE_ORDER, grid_range=GRID_RANGE,
        )

        train_kwargs = {
            "lr":           lr,
            "weight_decay": weight_decay,
            "reg_fn":       sparse_kan_edge_l1,
            "reg_weight":   reg_weight,
            "n_epochs":     300,
            "patience":     20,
        }
        if target_type == "continuous":
            train_kwargs["huber_delta"] = huber_delta

        return model, train_kwargs
    return factory


def robust_optimize(study, objective, n_trials, max_retries=3):
    """Retry study.optimize() on transient SQLite/Drive I/O errors."""
    for attempt in range(max_retries):
        try:
            study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
            return
        except Exception as e:
            if "disk I/O error" in str(e) or "OperationalError" in str(e):
                print(f"  ⚠ Transient storage error (attempt {attempt+1}/{max_retries}): {e}")
                time.sleep(5)
                continue
            raise  # a genuinely different error -- don't swallow it
    raise RuntimeError(f"study.optimize failed after {max_retries} retries -- likely a persistent Drive/mount issue, not transient")


# ═══════════════════════════════════════════════════════════════════════════
# RUN ONE PROBE: enqueue v1's best as trial 0, then 15 new trials
# ═══════════════════════════════════════════════════════════════════════════

def run_probe(cfg):
    dataset, split_name, target_type = cfg["dataset"], cfg["split"], cfg["target"]
    model_name = f"sparse_kan_{dataset}"
    label = cfg["label"]

    print(f"\n{'─'*70}")
    print(f"  PROBE {label}: {dataset} / {split_name} / {target_type} / seed={SEED}")
    print(f"  v1 val={cfg['v1_val']:.4f}   v2 val={cfg['v2_val']:.4f}   "
          f"delta={cfg['v2_val']-cfg['v1_val']:+.4f}"
          f"{'  [CONTROL]' if cfg.get('is_control') else ''}")
    print(f"{'─'*70}")

    set_seed(SEED)

    data         = load_split(split_name, dataset, SPLITS_DIR)
    feature_cols = data["feature_cols"]
    taxonomy_df  = taxonomy_dfs[dataset]

    n_pos      = data["y_train"].sum()
    n_neg      = len(data["y_train"]) - n_pos
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32)

    huber_delta = get_huber_delta(split_name) if target_type == "continuous" else None
    metric_name = "AUC" if target_type == "binary" else "R2"

    v1_best_params, v1_best_value = load_v1_best_params(model_name, target_type, split_name, SEED)
    print(f"  Loaded v1 best params for enqueue: {v1_best_params}")
    print(f"  (v1 stored best_value: {v1_best_value:.4f} -- should match table above)")

    study_path = RESULTS_DIR / "optuna" / f"{model_name}_{target_type}_{split_name}_v3probe.db"
    study_path.parent.mkdir(parents=True, exist_ok=True)

    study = optuna.create_study(
        study_name=f"{model_name}_{target_type}_{split_name}_v3probe_seed{SEED}",
        storage=f"sqlite:///{study_path}",
        direction="maximize",
        load_if_exists=True,
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )

    factory = make_model_factory_with_l1(feature_cols, taxonomy_df, huber_delta, target_type)

    def objective(trial):
        model, train_kwargs = factory(trial)
        batch_size = trial.params["batch_size"]

        loaders = get_dataloaders(
            split_name, dataset, SPLITS_DIR,
            target_type=target_type,
            batch_size=batch_size,
        )

        result = train_model(
            model=model,
            train_loader=loaders["train"],
            val_loader=loaders["val"],
            device=device,
            target_type=target_type,
            pos_weight=pos_weight if target_type == "binary" else None,
            verbose=False,
            **train_kwargs,
        )

        return result["best_val_metric"]

    optuna.logging.set_verbosity(optuna.logging.WARNING)

    existing = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)

    if existing == 0:
        # Trial 0: v1's exact best params, guaranteeing v3 >= v2 by construction
        study.enqueue_trial(v1_best_params)

    remaining = max(0, (N_TRIALS + 1) - existing)  # +1 for the enqueued trial
    if remaining > 0:
        print(f"  Running {remaining} trials (including 1 enqueued v1-params trial, "
              f"{existing} already complete)")
        robust_optimize(study, objective, remaining)
    else:
        print(f"  Study already has {existing} completed trials — skipping search")

    v3_best_params = study.best_params
    v3_best_value  = study.best_value

    print(f"\n  v3 best {metric_name}: {v3_best_value:.4f}  params: {v3_best_params}")

    delta_from_v1 = v3_best_value - cfg["v1_val"]
    recovered = abs(delta_from_v1) <= 0.03

    print(f"  v3 vs v1: {delta_from_v1:+.4f}   "
          f"-> {'RECOVERED (within 0.03)' if recovered else 'NOT recovered'}")

    return {
        "label": label, "dataset": dataset, "split": split_name, "target": target_type,
        "v1_val": cfg["v1_val"], "v2_val": cfg["v2_val"], "v3_val": v3_best_value,
        "delta_v1_v3": delta_from_v1, "recovered": recovered,
        "is_control": cfg.get("is_control", False),
        "v3_best_params": v3_best_params,
    }


# %% [markdown]
# ## Run All Five Probes

# %%
probe_results = []
total_start = time.time()

for cfg in PROBE_CONFIGS:
    try:
        res = run_probe(cfg)
        probe_results.append(res)
    except Exception as e:
        print(f"\n  ✗ PROBE {cfg['label']} FAILED: {e}")
        import traceback
        traceback.print_exc()
        continue

total_time = time.time() - total_start
print(f"\n\nTotal probe time: {total_time/60:.1f} minutes")


# %% [markdown]
# ## Decision Summary

# %%
print("=" * 90)
print("  V1 -> V2 -> V3 SUMMARY")
print("=" * 90)
print(f"\n  {'Config':<35} {'v1':>7} {'v2':>7} {'v3':>7} {'v3-v1':>8} {'Recovered?':>12}")
print("  " + "-" * 90)

non_control_recovered = 0
non_control_total = 0

for r in probe_results:
    flag = " [CONTROL]" if r["is_control"] else ""
    print(f"  {r['label']:<35} {r['v1_val']:>7.4f} {r['v2_val']:>7.4f} "
          f"{r['v3_val']:>7.4f} {r['delta_v1_v3']:>+8.4f} "
          f"{'YES' if r['recovered'] else 'no':>12}{flag}")
    if not r["is_control"]:
        non_control_total += 1
        if r["recovered"]:
            non_control_recovered += 1

control = next((r for r in probe_results if r["is_control"]), None)

print("\n" + "=" * 90)
print(f"  Non-control probes recovered: {non_control_recovered}/{non_control_total}")
if control:
    print(f"  Control probe delta (v3 vs v1): {control['delta_v1_v3']:+.4f}  "
          f"({'stable, as expected' if abs(control['delta_v1_v3']) <= 0.03 else 'moved MORE than expected -- treat non-control recoveries with extra caution, noise may be dominating'})")

print()
if non_control_total > 0 and non_control_recovered / non_control_total >= 0.75:
    print("  VERDICT: >=3/4 non-control probes recovered to within 0.03 of v1.")
    print("  A full re-search sweep (all 48 configs) is likely worth the overnight run.")
else:
    print("  VERDICT: fewer than 3/4 non-control probes recovered.")
    print("  Re-searching hyperparameters does not appear to close the v1->v2 gap")
    print("  for these configs. This is a legitimate finding: the fix is verified")
    print("  numerically (activations/masking/init all correct) but does not")
    print("  translate into predictive improvement here at the given search budget.")

print("\n  CAVEAT: 4 of 5 probes were selected on the LARGEST v1->v2 drops,")
print("  which are subject to regression toward the mean on re-measurement")
print("  independent of any causal hyperparameter effect. Treat this as a")
print("  fast diagnostic, not a definitive proof either way.")
print("=" * 90)

results_df = pd.DataFrame(probe_results)
results_df.to_csv(RESULTS_DIR / "v3_probe_summary.csv", index=False)
print(f"\n  Saved to {RESULTS_DIR / 'v3_probe_summary.csv'}")

# %% [markdown]
# ## Disconnect Runtime

# %%
print("Probe complete. Disconnecting runtime...")
from google.colab import runtime
runtime.unassign()

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4
SANITY CHECK: confirming init fix + eps=1e-5/affine=False
  ✓ eps=1e-5, affine=False on both bn1/bn2
  ✓ init fix on base_weight: ratio = 0.873
  ✓ init fix on spline_scaler: ratio = 0.973
  ✓ All masked parameters are exactly zero and finite
  ✓ forward pass OK

SANITY CHECK PASSED
Device: Tesla T4 (CUDA)

──────────────────────────────────────────────────────────────────────
  PROBE 1_afm_SplitB_binary: agg_full_moments / Split_B / binary / seed=42
  v1 val=0.7010   v2 val=0.5232   delta=-0.1778
──────────────────────────────────────────────────────────────────────
  Loaded v1 best params for enqueue: {'lr': 0.0005819088321713006, 'weight_decay': 4.66370557210144e-05, 'batch_siz

In [1]:
# %% [markdown]
# # Part 2 — Remaining 43 Probes (all configs, all 3 seeds)
#
# Extends the 5 hand-picked probes above to the full 48-tuple grid:
# 16 unique (dataset, target, split) combinations x 3 seeds = 48 total.
# The 5 probes already run above are automatically skipped (matched by
# dataset/split/target/seed against v3_probe_summary.csv).
#
# v1_val and v2_val are read directly from each config's saved checkpoint
# (best_val_metric, stored by save_checkpoint via train_model's return
# dict) in the ORIGINAL sparse_kan/ and sparse_kan_v2/ results
# directories -- no manual transcription for 43 configs.
#
# SAME with-L1-only search space, SAME enqueue_trial(v1_best_params) as
# trial 0, SAME decision rule (recovered = within 0.03 of v1_val).
#
# RESULTS: appended to the SAME v3_probe_summary.csv used by the 5
# original probes -- saved incrementally, one row per completed config.
#
# ROBUSTNESS (new in this version, after a Drive/SQLite disk I/O error
# corrupted a prior run): a Drive health check runs before anything else;
# both Optuna's SQLite writes AND the summary CSV append are wrapped in
# retry logic; the retry loop recomputes the trial count fresh on each
# attempt rather than blindly re-requesting the original count (which
# could overshoot the intended budget after a partial failure).
#
# Estimated runtime: 43 configs x ~5-8 min/config ~= 4-6 hours. Likely
# spans multiple Colab sessions -- safe to stop and re-run at any point,
# already-completed probes are skipped automatically.

# %%
# ── FRESH COLAB MOUNT ──
!pip install -q git+https://github.com/Blealtan/efficient-kan.git optuna

from google.colab import drive
drive.mount("/content/drive")

import sys
sys.path.insert(0, "/content/drive/MyDrive/Thesis/Code")

# %%
import json
import os
import numpy as np
import pandas as pd
import random
import time
import torch
import optuna
from pathlib import Path

from data_utils import load_split, get_dataloaders, get_device, load_theme_assignment
from training import train_model, save_checkpoint
from evaluation import evaluate_model, save_predictions, compute_calibration
from sparse_kan import SparseKAN, sparse_kan_edge_l1

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# ═══════════════════════════════════════════════════════════════════════════
# DRIVE HEALTH CHECK — confirm the mount can actually write/read before
# spending any GPU time. A prior run hit a "disk I/O error" from Optuna's
# SQLite backend writing to a FUSE-mounted Drive path; this catches that
# class of problem immediately rather than 40 minutes into a sweep.
# ═══════════════════════════════════════════════════════════════════════════

print("=" * 90)
print("DRIVE HEALTH CHECK")
print("=" * 90)

_test_path = "/content/drive/MyDrive/Thesis/_drive_health_check.txt"
try:
    with open(_test_path, "w") as f:
        f.write("health check")
    with open(_test_path, "r") as f:
        _content = f.read()
    os.remove(_test_path)
    assert _content == "health check"
    print("  ✓ Drive write/read/delete OK")
except Exception as e:
    raise RuntimeError(
        f"Drive mount is NOT healthy: {e}\n"
        f"Disconnect this runtime entirely (Runtime -> Disconnect and "
        f"delete runtime), then reconnect and re-mount before proceeding. "
        f"Do not attempt the sweep against an unhealthy mount."
    )

del _test_path

print("=" * 90)


# ═══════════════════════════════════════════════════════════════════════════
# SANITY CHECK — confirming init fix + eps=1e-5/affine=False
# ═══════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 90)
print("SANITY CHECK: confirming init fix + eps=1e-5/affine=False")
print("=" * 90)

_fake_tax = pd.DataFrame({
    "column":        ["f1", "f2", "f3", "f4", "f5"],
    "subtheme_id":   ["01_01"]*5,
    "subtheme_name": ["SubA"]*5,
    "theme_id":      ["01"]*5,
    "theme_name":    ["ThemeX"]*5,
})
_fcols = ["f1", "f2", "f3", "f4", "f5"]
N_DUMMY = len(_fcols)

_m = SparseKAN.from_taxonomy(_fake_tax, _fcols, grid_size=5, spline_order=3,
                             grid_range=[-1, 1])

assert abs(_m.bn1.eps - 1e-5) < 1e-9, f"bn1.eps={_m.bn1.eps}, expected 1e-5. STOP."
assert abs(_m.bn2.eps - 1e-5) < 1e-9, f"bn2.eps={_m.bn2.eps}, expected 1e-5. STOP."
assert _m.bn1.affine is False, f"bn1.affine={_m.bn1.affine}, expected False. STOP."
assert _m.bn2.affine is False, f"bn2.affine={_m.bn2.affine}, expected False. STOP."
print(f"  ✓ eps=1e-5, affine=False on both bn1/bn2")

_active = _m.layer0.mask[0].bool()
_fan_in = int(_active.sum().item())
_bound = 1.0 / _fan_in**0.5
for _pname, _W in [("base_weight",   _m.layer0.base_weight.data),
                   ("spline_scaler", _m.layer0.spline_scaler.data)]:
    _ratio = _W[0][_active].abs().max().item() / _bound
    assert _ratio > 0.5, f"{_pname} ratio={_ratio:.3f} -- STALE FILE. STOP."
print(f"  ✓ init fix confirmed on base_weight and spline_scaler")

assert _m.verify_masking(), "masking broken at construction. STOP."
_m.train()
assert _m(torch.randn(16, N_DUMMY)).shape == (16, 1)
print("  ✓ forward pass OK")

del _fake_tax, _fcols, _m, _active, _fan_in, _bound, N_DUMMY

print("\n" + "=" * 90)
print("SANITY CHECK PASSED")
print("=" * 90)


# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════

SPLITS_DIR  = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/04_splits")
THEMES_DIR  = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/05_themes")
HUBER_DELTA_PATH = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/03_targets/huber_delta.json")

V1_RESULTS_DIR = Path("/content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/sparse_kan")
V2_RESULTS_DIR = Path("/content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/sparse_kan_v2")
RESULTS_DIR    = Path("/content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/sparse_kan_v3_probe")

GRID_SIZE    = 14
SPLINE_ORDER = 3
GRID_RANGE   = [-5.5, 5.5]
N_TRIALS     = 15

DATASETS     = ["agg_full_moments", "agg_means"]
TARGET_TYPES = ["binary", "continuous"]
ALL_SPLITS   = ["Split_A", "Split_B", "Split_C", "Split_D"]
SEEDS        = [42, 123, 456]

with open(HUBER_DELTA_PATH) as f:
    HUBER_DELTAS = json.load(f)["deltas"]

def get_huber_delta(split_name):
    return HUBER_DELTAS[f"{split_name}/market"]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

taxonomy_dfs = {ds: load_theme_assignment(ds, THEMES_DIR) for ds in DATASETS}
device = get_device()


# ═══════════════════════════════════════════════════════════════════════════
# ROBUST I/O HELPERS — retry transient Drive/SQLite errors
# ═══════════════════════════════════════════════════════════════════════════

def is_transient_io_error(e):
    msg = str(e)
    return "disk I/O error" in msg or "OperationalError" in msg or "database is locked" in msg


def robust_optimize(study, objective, n_trials_target, max_retries=5):
    """
    Run trials until the study has n_trials_target COMPLETE trials total
    (not n_trials_target MORE trials on every retry -- recomputed fresh
    each attempt so a partial failure can't cause overshoot).
    """
    for attempt in range(max_retries):
        existing = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
        remaining_now = max(0, n_trials_target - existing)
        if remaining_now == 0:
            return
        try:
            study.optimize(objective, n_trials=remaining_now, show_progress_bar=True)
            return
        except Exception as e:
            if is_transient_io_error(e) and attempt < max_retries - 1:
                print(f"  ⚠ Transient storage error (attempt {attempt+1}/{max_retries}): {e}")
                time.sleep(10)
                continue
            raise


def robust_append_csv(summary_path, row_df, max_retries=5):
    """Append one row to the summary CSV, retrying on transient Drive I/O errors."""
    for attempt in range(max_retries):
        try:
            if summary_path.exists():
                combined = pd.concat([pd.read_csv(summary_path), row_df], ignore_index=True)
            else:
                combined = row_df
            combined.to_csv(summary_path, index=False)
            return
        except Exception as e:
            if is_transient_io_error(e) and attempt < max_retries - 1:
                print(f"  ⚠ Transient error saving summary (attempt {attempt+1}/{max_retries}): {e}")
                time.sleep(10)
                continue
            raise


# ═══════════════════════════════════════════════════════════════════════════
# BUILD THE FULL 48-TUPLE GRID, THEN SUBTRACT WHAT'S ALREADY BEEN PROBED
# ═══════════════════════════════════════════════════════════════════════════

FULL_GRID = []
for seed in SEEDS:
    for dataset in DATASETS:
        for target_type in TARGET_TYPES:
            for split_name in ALL_SPLITS:
                FULL_GRID.append({
                    "dataset": dataset, "target": target_type,
                    "split": split_name, "seed": seed,
                })

print(f"Full grid: {len(FULL_GRID)} (dataset, target, split, seed) tuples")

summary_path = RESULTS_DIR / "v3_probe_summary.csv"
already_done = set()
if summary_path.exists():
    existing_df = pd.read_csv(summary_path)
    for _, row in existing_df.iterrows():
        seed_val = row["seed"] if "seed" in existing_df.columns and not pd.isna(row.get("seed")) else 42
        already_done.add((row["dataset"], row["target"], row["split"], seed_val))
    print(f"Found {len(already_done)} already-completed probes in {summary_path}")
else:
    print("No existing v3_probe_summary.csv found -- starting fresh")

REMAINING = [
    cfg for cfg in FULL_GRID
    if (cfg["dataset"], cfg["target"], cfg["split"], cfg["seed"]) not in already_done
]

print(f"Remaining to run: {len(REMAINING)}")
assert len(REMAINING) <= 43, (
    f"Expected at most 43 remaining, got {len(REMAINING)} -- check that "
    f"the 5 original probes were correctly detected as already-done."
)


# ═══════════════════════════════════════════════════════════════════════════
# LOAD v1_val / v2_val FROM SAVED CHECKPOINTS
# ═══════════════════════════════════════════════════════════════════════════

def get_val_metric_from_checkpoint(results_dir, seed, dataset, target_type, split_name):
    model_name = f"sparse_kan_{dataset}"
    ckpt_path = (results_dir / f"seed_{seed}" / "checkpoints"
                 / f"{model_name}_{target_type}_{split_name}.pt")
    if not ckpt_path.exists():
        return None
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    return ckpt.get("best_val_metric", None)


# ═══════════════════════════════════════════════════════════════════════════
# LOAD v1's BEST PARAMS — enqueue_trial starting point
# ═══════════════════════════════════════════════════════════════════════════

def load_v1_best_params(model_name, target_type, split_name, seed):
    study_dir = V1_RESULTS_DIR / f"seed_{seed}" / "optuna"
    path_l1 = study_dir / f"{model_name}_{target_type}_{split_name}_with_L1_seed{seed}.db"

    if not path_l1.exists():
        raise FileNotFoundError(f"v1 with-L1 study not found at {path_l1}")

    study = optuna.load_study(
        study_name=f"{model_name}_{target_type}_{split_name}_with_L1_seed{seed}",
        storage=f"sqlite:///{path_l1}",
    )
    return study.best_params, study.best_value


def make_model_factory_with_l1(feature_cols, taxonomy_df, huber_delta, target_type):
    def factory(trial):
        lr           = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
        batch_size   = trial.suggest_categorical("batch_size", [64, 128, 256])
        reg_weight   = trial.suggest_float("reg_weight", 1e-7, 1e-3, log=True)

        model = SparseKAN.from_taxonomy(
            taxonomy_df, feature_cols,
            grid_size=GRID_SIZE, spline_order=SPLINE_ORDER, grid_range=GRID_RANGE,
        )
        train_kwargs = {
            "lr": lr, "weight_decay": weight_decay,
            "reg_fn": sparse_kan_edge_l1, "reg_weight": reg_weight,
            "n_epochs": 300, "patience": 20,
        }
        if target_type == "continuous":
            train_kwargs["huber_delta"] = huber_delta
        return model, train_kwargs
    return factory


def run_probe(cfg):
    dataset, split_name, target_type, seed = (
        cfg["dataset"], cfg["split"], cfg["target"], cfg["seed"]
    )
    model_name = f"sparse_kan_{dataset}"
    label = f"{dataset}_{split_name}_{target_type}_seed{seed}"

    print(f"\n{'─'*70}")
    print(f"  PROBE {label}")
    print(f"{'─'*70}")

    v1_val = get_val_metric_from_checkpoint(V1_RESULTS_DIR, seed, dataset, target_type, split_name)
    v2_val = get_val_metric_from_checkpoint(V2_RESULTS_DIR, seed, dataset, target_type, split_name)

    if v1_val is None or v2_val is None:
        print(f"  ⚠ Missing checkpoint (v1_val={v1_val}, v2_val={v2_val}) -- SKIPPING")
        return None

    print(f"  v1 val={v1_val:.4f}   v2 val={v2_val:.4f}   delta={v2_val-v1_val:+.4f}")

    set_seed(seed)

    data         = load_split(split_name, dataset, SPLITS_DIR)
    feature_cols = data["feature_cols"]
    taxonomy_df  = taxonomy_dfs[dataset]

    n_pos      = data["y_train"].sum()
    n_neg      = len(data["y_train"]) - n_pos
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32)

    huber_delta = get_huber_delta(split_name) if target_type == "continuous" else None
    metric_name = "AUC" if target_type == "binary" else "R2"

    v1_best_params, v1_stored_value = load_v1_best_params(model_name, target_type, split_name, seed)

    study_path = RESULTS_DIR / "optuna" / f"{model_name}_{target_type}_{split_name}_seed{seed}_v3probe.db"
    study_path.parent.mkdir(parents=True, exist_ok=True)

    study = optuna.create_study(
        study_name=f"{model_name}_{target_type}_{split_name}_v3probe_seed{seed}",
        storage=f"sqlite:///{study_path}",
        direction="maximize",
        load_if_exists=True,
        sampler=optuna.samplers.TPESampler(seed=seed),
    )

    factory = make_model_factory_with_l1(feature_cols, taxonomy_df, huber_delta, target_type)

    def objective(trial):
        model, train_kwargs = factory(trial)
        batch_size = trial.params["batch_size"]
        loaders = get_dataloaders(split_name, dataset, SPLITS_DIR,
                                  target_type=target_type, batch_size=batch_size)
        result = train_model(
            model=model, train_loader=loaders["train"], val_loader=loaders["val"],
            device=device, target_type=target_type,
            pos_weight=pos_weight if target_type == "binary" else None,
            verbose=False, **train_kwargs,
        )
        return result["best_val_metric"]

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    existing = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)

    if existing == 0:
        study.enqueue_trial(v1_best_params)

    print(f"  Target: {N_TRIALS + 1} total trials (incl. 1 enqueued v1-params trial), "
          f"{existing} already complete")
    robust_optimize(study, objective, N_TRIALS + 1)

    v3_best_value = study.best_value
    delta_from_v1 = v3_best_value - v1_val
    recovered = abs(delta_from_v1) <= 0.03

    print(f"  v3 best {metric_name}: {v3_best_value:.4f}   "
          f"vs v1: {delta_from_v1:+.4f}   "
          f"-> {'RECOVERED' if recovered else 'NOT recovered'}")

    return {
        "label": label, "dataset": dataset, "split": split_name,
        "target": target_type, "seed": seed,
        "v1_val": v1_val, "v2_val": v2_val, "v3_val": v3_best_value,
        "delta_v1_v3": delta_from_v1, "recovered": recovered,
        "is_control": False,
    }


# %% [markdown]
# ## Run All Remaining Probes (incremental save after each one)

# %%
total_start = time.time()
n_done_this_run = 0

for cfg in REMAINING:
    try:
        res = run_probe(cfg)
        if res is None:
            continue

        row_df = pd.DataFrame([res])
        robust_append_csv(summary_path, row_df)

        n_done_this_run += 1
        elapsed = time.time() - total_start
        rate = elapsed / n_done_this_run
        remaining_est = rate * (len(REMAINING) - n_done_this_run)
        print(f"  ✓ {n_done_this_run}/{len(REMAINING)} done this session  "
              f"({elapsed/60:.1f}min elapsed, ~{remaining_est/60:.1f}min remaining)")

    except Exception as e:
        print(f"\n  ✗ PROBE FAILED: {cfg} -- {e}")
        import traceback
        traceback.print_exc()
        continue

print(f"\n\nTotal time this session: {(time.time()-total_start)/60:.1f} minutes")
print(f"All results (old + new) saved to {summary_path}")


# %% [markdown]
# ## Full Decision Summary (all 48 probes, old + new)

# %%
if summary_path.exists():
    full_df = pd.read_csv(summary_path)
    n_recovered = full_df["recovered"].sum()
    n_total = len(full_df)

    print("=" * 90)
    print(f"  FULL PROBE SUMMARY: {n_total}/48 probes completed")
    print(f"  Recovered (within 0.03 of v1): {n_recovered}/{n_total} "
          f"({n_recovered/n_total:.1%})")
    print("=" * 90)
    print(full_df[["label", "v1_val", "v2_val", "v3_val", "delta_v1_v3", "recovered"]]
          .to_string(index=False))
else:
    print(f"No summary file exists at {summary_path} -- no probes completed successfully yet.")

Mounted at /content/drive
PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4
DRIVE HEALTH CHECK
  ✓ Drive write/read/delete OK

SANITY CHECK: confirming init fix + eps=1e-5/affine=False
  ✓ eps=1e-5, affine=False on both bn1/bn2
  ✓ init fix confirmed on base_weight and spline_scaler
  ✓ All masked parameters are exactly zero and finite
  ✓ forward pass OK

SANITY CHECK PASSED
Device: Tesla T4 (CUDA)
Full grid: 48 (dataset, target, split, seed) tuples
Found 5 already-completed probes in /content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/sparse_kan_v3_probe/v3_probe_summary.csv
Remaining to run: 43

──────────────────────────────────────────────────────────────────────
  PROBE agg_full_moments_Split_A_binary_seed42
──────────────────────────────────────────────────────────────────────
  v1 val=0.7042   v2 val=0.6930   delta=-0.0112


[I 2026-08-15 17:49:13,809] Using an existing study with name 'sparse_kan_agg_full_moments_binary_Split_A_v3probe_seed42' instead of creating a new one.


  Target: 16 total trials (incl. 1 enqueued v1-params trial), 16 already complete
  v3 best AUC: 0.8681   vs v1: +0.1639   -> NOT recovered
  ✓ 1/43 done this session  (0.3min elapsed, ~11.8min remaining)

──────────────────────────────────────────────────────────────────────
  PROBE agg_full_moments_Split_C_binary_seed42
──────────────────────────────────────────────────────────────────────
  v1 val=0.7441   v2 val=0.6065   delta=-0.1376
  Target: 16 total trials (incl. 1 enqueued v1-params trial), 16 already complete
  v3 best AUC: 0.7593   vs v1: +0.0153   -> RECOVERED
  ✓ 2/43 done this session  (0.5min elapsed, ~10.0min remaining)

──────────────────────────────────────────────────────────────────────
  PROBE agg_full_moments_Split_D_binary_seed42
──────────────────────────────────────────────────────────────────────
  v1 val=0.7209   v2 val=0.6591   delta=-0.0618
  Target: 16 total trials (incl. 1 enqueued v1-params trial), 16 already complete
  v3 best AUC: 0.7302   vs v1: +0.00

  0%|          | 0/2 [00:00<?, ?it/s]

  v3 best AUC: 0.7082   vs v1: +0.0094   -> RECOVERED
  ✓ 23/43 done this session  (3.1min elapsed, ~2.7min remaining)

──────────────────────────────────────────────────────────────────────
  PROBE agg_means_Split_A_continuous_seed123
──────────────────────────────────────────────────────────────────────
  v1 val=0.1178   v2 val=-0.2318   delta=-0.3496
  Target: 16 total trials (incl. 1 enqueued v1-params trial), 0 already complete


...

  v3 best R2: 0.0971   vs v1: -0.0019   -> RECOVERED
  ✓ 41/43 done this session  (71.2min elapsed, ~3.5min remaining)

──────────────────────────────────────────────────────────────────────
  PROBE agg_means_Split_C_continuous_seed456
──────────────────────────────────────────────────────────────────────
  v1 val=0.2548   v2 val=0.1043   delta=-0.1505
  Target: 16 total trials (incl. 1 enqueued v1-params trial), 0 already complete


  0%|          | 0/16 [00:00<?, ?it/s]

  v3 best R2: 0.2760   vs v1: +0.0212   -> RECOVERED
  ✓ 42/43 done this session  (75.4min elapsed, ~1.8min remaining)

──────────────────────────────────────────────────────────────────────
  PROBE agg_means_Split_D_continuous_seed456
──────────────────────────────────────────────────────────────────────
  v1 val=0.1784   v2 val=-0.1375   delta=-0.3159
  Target: 16 total trials (incl. 1 enqueued v1-params trial), 0 already complete


  0%|          | 0/16 [00:00<?, ?it/s]

  v3 best R2: 0.2113   vs v1: +0.0328   -> NOT recovered
  ✓ 43/43 done this session  (79.0min elapsed, ~0.0min remaining)


Total time this session: 79.0 minutes
All results (old + new) saved to /content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/sparse_kan_v3_probe/v3_probe_summary.csv
  FULL PROBE SUMMARY: 48/48 probes completed
  Recovered (within 0.03 of v1): 18/48 (37.5%)
                                      label    v1_val    v2_val   v3_val  delta_v1_v3  recovered
                        1_afm_SplitB_binary  0.701000  0.523200 0.777646     0.076646      False
                      2_means_SplitB_binary  0.812500  0.588800 0.761997    -0.050503      False
                    3_afm_SplitC_continuous  0.283000  0.060000 0.186326    -0.096674      False
                  4_means_SplitC_continuous  0.283200  0.001100 0.221869    -0.061331      False
              5_CONTROL_means_SplitC_binary  0.759300  0.745300 0.751701    -0.007599       True
     agg_full_moments_S

In [2]:
# %% [markdown]
# # 04 — Sparse KAN v3: Final Retrain (converts probe search results into
# #      comparable retrain metrics, test metrics, and backtests)
#
# ═══════════════════════════════════════════════════════════════════════
# WHY THIS NOTEBOOK EXISTS
# ═══════════════════════════════════════════════════════════════════════
#
# The v3 probe notebook produced, for each config, `study.best_value` --
# the MAXIMUM validation metric over 16 Optuna trials. That is an order
# statistic, not a performance measurement, and it is NOT comparable to
# v1_val / v2_val, both of which are FINAL-RETRAIN validation metrics
# recorded by train_model().
#
# The gap is not small. In the original v1 sweep, Optuna's reported best
# exceeded the achieved retrain val in 24/24 configs, by ~0.05 on average
# (e.g. sparse_kan/agg_full_moments/Split_B/binary/seed42: Optuna best
# 0.8058 vs final retrain val 0.7010 -- a 0.105 gap on the same config).
#
# So the v3 probe summary CANNOT be read as "v3 beat v1". This notebook
# fixes that by doing what v2 did: take the selected hyperparameters,
# retrain once, and record the same quantities on the same footing.
#
# AFTER THIS RUNS you will have, per config, on a like-for-like basis:
#     v1_val, v2_val, v3_val   (all final-retrain)
#     v1_test, v2_test, v3_test
# plus v3 backtests.
#
# ═══════════════════════════════════════════════════════════════════════
# BUDGET ASYMMETRY -- STATE THIS IN THE METHODS SECTION
# ═══════════════════════════════════════════════════════════════════════
#
#   v1: pre-fix code,  70 trials searched (30 no-L1 + 40 with-L1)
#   v2: fixed code,     0 trials (v1's hyperparameters held fixed)
#   v3: fixed code,    16 trials searched (with-L1 only, v1 params enqueued)
#
# v3 received a SMALLER search budget than v1. Any v3 improvement over v1
# is therefore a lower bound; any v3 shortfall against v1 is confounded
# with budget and must not be read as evidence about the architecture.
# The v1 -> v2 comparison is the clean controlled experiment (identical
# hyperparameters, identical seeds, only init + eps changed) and is
# unaffected by any of this.
#
# Estimated runtime: ~20 minutes for all 48 configs (retrain only).
# %%
# ── COLAB SETUP ──
!pip install -q git+https://github.com/Blealtan/efficient-kan.git optuna
from google.colab import drive
drive.mount("/content/drive")
import sys
sys.path.insert(0, "/content/drive/MyDrive/Thesis/Code")

# %%
import json
import os
import numpy as np
import pandas as pd
import random
import time
import torch
import optuna
from pathlib import Path

from data_utils import load_split, get_dataloaders, get_device, load_theme_assignment
from training import train_model, save_checkpoint
from evaluation import (
    evaluate_model, save_predictions, compute_calibration,
    load_predictions, run_full_backtest,
)
from sparse_kan import SparseKAN, sparse_kan_edge_l1

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# ═══════════════════════════════════════════════════════════════════════
# DRIVE HEALTH CHECK
# ═══════════════════════════════════════════════════════════════════════
print("=" * 90)
print("DRIVE HEALTH CHECK")
print("=" * 90)
_test_path = "/content/drive/MyDrive/Thesis/_drive_health_check.txt"
try:
    with open(_test_path, "w") as f:
        f.write("health check")
    with open(_test_path, "r") as f:
        _content = f.read()
    os.remove(_test_path)
    assert _content == "health check"
    print("  ✓ Drive write/read/delete OK")
except Exception as e:
    raise RuntimeError(
        f"Drive mount is NOT healthy: {e}\n"
        f"Disconnect the runtime entirely (Runtime -> Disconnect and delete "
        f"runtime), reconnect, re-mount, then retry."
    )
del _test_path


# ═══════════════════════════════════════════════════════════════════════
# SANITY CHECK — the live sparse_kan.py must have BOTH fixes
# ═══════════════════════════════════════════════════════════════════════
print("\n" + "=" * 90)
print("SANITY CHECK: init fix + eps=1e-5 / affine=False")
print("=" * 90)

# Fixture note: in_features MUST be >> fan_in or the init bug is invisible
# (the bug substituted 1/sqrt(in_features) for 1/sqrt(fan_in)).
N_DUMMY = 200
_cols = [f"f{i}" for i in range(N_DUMMY)]
_fake_tax = pd.DataFrame({
    "column":        _cols,
    "subtheme_id":   ["01_01"]*5 + [f"02_{i:02d}" for i in range(N_DUMMY - 5)],
    "subtheme_name": ["SubA"]*5  + ["SubB"]*(N_DUMMY - 5),
    "theme_id":      ["01"]*5    + ["02"]*(N_DUMMY - 5),
    "theme_name":    ["ThemeX"]*5 + ["ThemeY"]*(N_DUMMY - 5),
})
_m = SparseKAN.from_taxonomy(_fake_tax, _cols, grid_size=5, spline_order=3,
                             grid_range=[-1, 1])

assert abs(_m.bn1.eps - 1e-5) < 1e-9, f"bn1.eps={_m.bn1.eps}, expected 1e-5. STOP."
assert abs(_m.bn2.eps - 1e-5) < 1e-9, f"bn2.eps={_m.bn2.eps}, expected 1e-5. STOP."
assert _m.bn1.affine is False and _m.bn2.affine is False, "affine != False. STOP."
print("  ✓ eps=1e-5, affine=False on both bn1/bn2")

_active = _m.layer0.mask[0].bool()
_fan_in = int(_active.sum().item())
assert _fan_in == 5, f"fixture built wrong: fan_in={_fan_in}"
_bound = 1.0 / _fan_in**0.5
# E[max of n draws] = bound * n/(n+1) = 0.833 when fixed; ~0.13 when broken
for _pname, _W in [("base_weight",   _m.layer0.base_weight.data),
                   ("spline_scaler", _m.layer0.spline_scaler.data)]:
    _ratio = _W[0][_active].abs().max().item() / _bound
    assert _ratio > 0.5, (
        f"{_pname} ratio={_ratio:.3f} -- expected ~0.83 (fixed) vs ~0.13 "
        f"(broken). sparse_kan.py on Drive is STALE. STOP."
    )
    print(f"  ✓ init fix on {_pname}: ratio = {_ratio:.3f} (expect ~0.83)")

assert _m.verify_masking(), "masking broken at construction. STOP."
_m.train()
assert _m(torch.randn(16, N_DUMMY)).shape == (16, 1)
print("  ✓ forward pass OK in train mode")

del _fake_tax, _cols, _m, _active, _fan_in, _bound, N_DUMMY
print("\n" + "=" * 90)
print("SANITY CHECK PASSED")
print("=" * 90)


# ═══════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════
SPLITS_DIR  = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/04_splits")
THEMES_DIR  = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/05_themes")
HUBER_DELTA_PATH = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/03_targets/huber_delta.json")

BASE = Path("/content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2")
V1_RESULTS_DIR    = BASE / "sparse_kan"            # read only
V2_RESULTS_DIR    = BASE / "sparse_kan_v2"         # read only
V3_PROBE_DIR      = BASE / "sparse_kan_v3_probe"   # read only (Optuna .db)
RESULTS_DIR       = BASE / "sparse_kan_v3"         # WRITE here

DATASETS     = ["agg_full_moments", "agg_means"]
TARGET_TYPES = ["binary", "continuous"]
ALL_SPLITS   = ["Split_A", "Split_B", "Split_C", "Split_D"]
SEEDS        = [42, 123, 456]

GRID_SIZE    = 14
SPLINE_ORDER = 3
GRID_RANGE   = [-5.5, 5.5]
ACTIVATION_PROBE_N = 2048

with open(HUBER_DELTA_PATH) as f:
    HUBER_DELTAS = json.load(f)["deltas"]

def get_huber_delta(split_name):
    return HUBER_DELTAS[f"{split_name}/market"]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

taxonomy_dfs = {ds: load_theme_assignment(ds, THEMES_DIR) for ds in DATASETS}
device = get_device()

print("\nTaxonomies:")
for ds, df in taxonomy_dfs.items():
    print(f"  {ds}: {len(df)} features, {df['subtheme_id'].nunique()} subthemes, "
          f"{df['theme_id'].nunique()} themes")


# ═══════════════════════════════════════════════════════════════════════
# ROBUST I/O
# ═══════════════════════════════════════════════════════════════════════
def is_transient_io_error(e):
    msg = str(e)
    return ("disk I/O error" in msg or "OperationalError" in msg
            or "database is locked" in msg)

def robust_append_csv(path, row_df, max_retries=5):
    for attempt in range(max_retries):
        try:
            combined = (pd.concat([pd.read_csv(path), row_df], ignore_index=True)
                        if path.exists() else row_df)
            path.parent.mkdir(parents=True, exist_ok=True)
            combined.to_csv(path, index=False)
            return
        except Exception as e:
            if is_transient_io_error(e) and attempt < max_retries - 1:
                print(f"  ⚠ Transient CSV error (attempt {attempt+1}): {e}")
                time.sleep(10)
                continue
            raise


# ═══════════════════════════════════════════════════════════════════════
# PRE-FLIGHT: every v3 probe study must exist and be fully searched
# ═══════════════════════════════════════════════════════════════════════
N_TRIALS_EXPECTED = 16   # 15 searched + 1 enqueued v1-params trial

def v3_study_path(model_name, target_type, split_name, seed):
    return (V3_PROBE_DIR / "optuna"
            / f"{model_name}_{target_type}_{split_name}_seed{seed}_v3probe.db")

def v3_study_name(model_name, target_type, split_name, seed):
    return f"{model_name}_{target_type}_{split_name}_v3probe_seed{seed}"

print("\n" + "=" * 90)
print("PRE-FLIGHT: auditing v3 probe studies")
print("=" * 90)

_missing, _thin = [], []
for _seed in SEEDS:
    for _ds in DATASETS:
        for _tt in TARGET_TYPES:
            for _sp in ALL_SPLITS:
                _mn = f"sparse_kan_{_ds}"
                _p = v3_study_path(_mn, _tt, _sp, _seed)
                _nm = v3_study_name(_mn, _tt, _sp, _seed)
                if not _p.exists():
                    _missing.append(_nm)
                    continue
                try:
                    _st = optuna.load_study(study_name=_nm, storage=f"sqlite:///{_p}")
                    _n = sum(1 for t in _st.trials
                             if t.state == optuna.trial.TrialState.COMPLETE)
                    if _n < N_TRIALS_EXPECTED:
                        _thin.append((_nm, _n))
                except Exception as e:
                    _missing.append(f"{_nm} (load failed: {e})")

print(f"  missing studies:        {len(_missing)}")
for _x in _missing[:15]:
    print(f"    {_x}")
print(f"  under-searched studies: {len(_thin)}  (expected {N_TRIALS_EXPECTED} trials)")
for _x in _thin[:15]:
    print(f"    {_x[0]}  ({_x[1]}/{N_TRIALS_EXPECTED})")

if _missing or _thin:
    print("\n  ⚠ Configs with missing/short studies will still retrain using "
          "whatever best_params exist, but their search budget differs from "
          "the rest. Note them in the write-up, or re-run the probe for them.")
else:
    print("\n  ✓ All 48 v3 studies present and fully searched.")
del _missing, _thin


# ═══════════════════════════════════════════════════════════════════════
# HELPERS
# ═══════════════════════════════════════════════════════════════════════
def load_v3_best_params(model_name, target_type, split_name, seed):
    """Best params + search-max value from the v3 probe study. Raises loudly
    rather than silently falling back to a fresh search."""
    p = v3_study_path(model_name, target_type, split_name, seed)
    if not p.exists():
        raise FileNotFoundError(f"v3 probe study not found at {p}")
    study = optuna.load_study(
        study_name=v3_study_name(model_name, target_type, split_name, seed),
        storage=f"sqlite:///{p}",
    )
    n_complete = sum(1 for t in study.trials
                     if t.state == optuna.trial.TrialState.COMPLETE)
    return study.best_params, study.best_value, n_complete


def get_ckpt_metrics(results_dir, seed, dataset, target_type, split_name):
    """Retrain val metric + test metrics from a saved checkpoint / metrics file."""
    model_name = f"sparse_kan_{dataset}"
    ckpt_path = (results_dir / f"seed_{seed}" / "checkpoints"
                 / f"{model_name}_{target_type}_{split_name}.pt")
    if not ckpt_path.exists():
        return None
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    return ckpt.get("best_val_metric", None)


def make_activation_callback(probe_batch, device):
    @torch.no_grad()
    def callback(model, epoch):
        model.eval()
        x0 = probe_batch.to(device)
        x1 = model.layer0(x0)
        x1n = model.bn1(x1)
        x1c = torch.clamp(x1n, min=-5.0, max=5.0)
        x2 = model.layer1(x1c)
        x2n = model.bn2(x2)
        model.train()
        c1 = (x1n.abs() > 5.0).float().mean().item()
        c2 = (x2n.abs() > 5.0).float().mean().item()
        print(f"    [act @ ep {epoch}]  "
              f"L1 pre-BN std={x1.std().item():6.3f} post-BN std={x1n.std().item():5.3f} "
              f"clamp={c1:.3%}  |  "
              f"L2 pre-BN std={x2.std().item():6.3f} post-BN std={x2n.std().item():5.3f} "
              f"clamp={c2:.3%}")
    return callback


# ═══════════════════════════════════════════════════════════════════════
# SINGLE RETRAIN
# ═══════════════════════════════════════════════════════════════════════
def run_single_retrain(split_name, dataset, target_type, seed, seed_results_dir):
    model_name = f"sparse_kan_{dataset}"

    print(f"\n{'─'*60}")
    print(f"  seed={seed} / {dataset} / {split_name} / {target_type}")
    print(f"{'─'*60}")

    data         = load_split(split_name, dataset, SPLITS_DIR)
    feature_cols = data["feature_cols"]
    taxonomy_df  = taxonomy_dfs[dataset]

    n_pos      = data["y_train"].sum()
    n_neg      = len(data["y_train"]) - n_pos
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32)
    huber_delta = get_huber_delta(split_name) if target_type == "continuous" else None

    best_params, v3_search_max, n_trials = load_v3_best_params(
        model_name, target_type, split_name, seed)
    print(f"  v3 params ({n_trials} trials, search max={v3_search_max:.4f}): {best_params}")

    v1_val = get_ckpt_metrics(V1_RESULTS_DIR, seed, dataset, target_type, split_name)
    v2_val = get_ckpt_metrics(V2_RESULTS_DIR, seed, dataset, target_type, split_name)

    rng = np.random.default_rng(seed)
    n_avail = data["X_train"].shape[0]
    probe_idx = rng.choice(n_avail, size=min(ACTIVATION_PROBE_N, n_avail), replace=False)
    activation_probe = torch.tensor(data["X_train"][probe_idx], dtype=torch.float32)

    batch_size = best_params.get("batch_size", 128)
    loaders = get_dataloaders(split_name, dataset, SPLITS_DIR,
                              target_type=target_type, batch_size=batch_size)

    model = SparseKAN.from_taxonomy(
        taxonomy_df, feature_cols,
        grid_size=GRID_SIZE, spline_order=SPLINE_ORDER, grid_range=GRID_RANGE,
    )

    # The v3 probe searched the with-L1 space only, so reg is always applied.
    train_kwargs = {
        "lr":             best_params["lr"],
        "weight_decay":   best_params["weight_decay"],
        "reg_fn":         sparse_kan_edge_l1,
        "reg_weight":     best_params["reg_weight"],
        "pos_weight":     pos_weight if target_type == "binary" else None,
        "n_epochs":       300,
        "patience":       20,
        "verbose":        True,
        "log_every":      20,
        "epoch_callback": make_activation_callback(activation_probe, device),
    }
    if target_type == "continuous":
        train_kwargs["huber_delta"] = huber_delta

    result = train_model(
        model=model, train_loader=loaders["train"], val_loader=loaders["val"],
        device=device, target_type=target_type, **train_kwargs,
    )

    assert model.verify_masking(), (
        "Masked weights non-zero after training -- structural sparsity broken."
    )

    all_metrics = {}
    for part in ["train", "val", "test"]:
        metrics = evaluate_model(
            model, loaders[part], device, target_type,
            y_true_binary=data[f"y_{part}"] if target_type == "continuous" else None,
        )
        if target_type == "binary":
            metrics["ece"] = compute_calibration(metrics["y_true"], metrics["y_prob"])["ece"]

        save_predictions(
            model_name=model_name, split_name=split_name,
            target_type=target_type, part=part,
            dates=data[f"dates_{part}"], returns=data[f"returns_{part}"],
            metrics=metrics,
            hyperparameters=(
                {**best_params, "seed": seed, "huber_delta": huber_delta,
                 "arm": "v3", "v3_search_max": v3_search_max,
                 "n_search_trials": n_trials}
                if part == "test" else None
            ),
            results_dir=seed_results_dir,
            y_true_binary=data[f"y_{part}"] if target_type == "continuous" else None,
        )
        all_metrics[part] = metrics

    ckpt_dir = seed_results_dir / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    save_checkpoint(
        model=model, train_result=result,
        hyperparameters={**best_params, "seed": seed, "huber_delta": huber_delta,
                         "arm": "v3", "v3_search_max": v3_search_max},
        model_config={
            "type": "SparseKAN_Masked_BN_v3", "dataset": dataset,
            "target_type": target_type, "n_features": data["n_features"],
            "n_subthemes": model.n_subthemes, "n_themes": model.n_themes,
            "grid_size": GRID_SIZE, "spline_order": SPLINE_ORDER,
            "grid_range": GRID_RANGE, "eps": 1e-5,
            "active_edges": model.count_active_edges(),
            "total_edges": model.count_total_edges(),
            "active_parameters": model.count_active_parameters(),
        },
        path=ckpt_dir / f"{model_name}_{target_type}_{split_name}.pt",
    )

    v3_val = result["best_val_metric"]

    if target_type == "binary":
        print(f"\n  Results (v3, seed={seed}):")
        print(f"    Train AUC: {all_metrics['train']['auc']:.4f}")
        print(f"    Val AUC:   {v3_val:.4f}")
        print(f"    Test AUC:  {all_metrics['test']['auc']:.4f}")
    else:
        print(f"\n  Results (v3, seed={seed}, huber_delta={huber_delta:.4f}):")
        print(f"    Train R2: {all_metrics['train']['r2']:.4f}")
        print(f"    Val R2:   {v3_val:.4f}")
        print(f"    Test R2:  {all_metrics['test']['r2']:.4f}  "
              f"(MSE={all_metrics['test']['mse']:.4f})")
        print(f"    Derived AUC: {all_metrics['test']['derived_auc']:.4f}")

    # LIKE-FOR-LIKE: all three are final-retrain validation metrics.
    print(f"    ── retrain val:  v1={v1_val if v1_val is None else f'{v1_val:.4f}'}"
          f"   v2={v2_val if v2_val is None else f'{v2_val:.4f}'}"
          f"   v3={v3_val:.4f}")
    print(f"    ── (v3 Optuna search max was {v3_search_max:.4f} -- NOT comparable)")

    row = {
        "seed": seed, "dataset": dataset, "split": split_name, "target": target_type,
        "v1_val_retrain": v1_val, "v2_val_retrain": v2_val, "v3_val_retrain": v3_val,
        "v3_search_max": v3_search_max, "n_search_trials": n_trials,
        "best_epoch": result["best_epoch"], "time_s": result["total_time"],
        **{f"test_{k}": v for k, v in all_metrics["test"].items()
           if not isinstance(v, np.ndarray)},
        **{f"train_{k}": v for k, v in all_metrics["train"].items()
           if not isinstance(v, np.ndarray)},
    }
    return row


# %% [markdown]
# ## Run the v3 retrain sweep (48 configs, retrain only, ~20 min)

# %%
total_runs  = len(SEEDS) * len(DATASETS) * len(TARGET_TYPES) * len(ALL_SPLITS)
row_path    = RESULTS_DIR / "all_seeds_raw.csv"

print("=" * 70)
print(f"  SPARSE KAN v3 (RETRAIN-ONLY): {total_runs} runs")
print(f"  Hyperparameters from: {V3_PROBE_DIR}/optuna/  (16-trial search)")
print(f"  Writing NEW results to: {RESULTS_DIR}")
print(f"  v1 and v2 directories are NEVER written to.")
print("=" * 70)

# resume support
already = set()
if row_path.exists():
    _prev = pd.read_csv(row_path)
    for _, r in _prev.iterrows():
        already.add((r["seed"], r["dataset"], r["target"], r["split"]))
    print(f"  Resuming: {len(already)} configs already retrained")

all_results = []
completed = failed = 0
total_start = time.time()

for seed in SEEDS:
    set_seed(seed)
    seed_results_dir = RESULTS_DIR / f"seed_{seed}"
    print(f"\n\n{'═'*70}")
    print(f"  SEED {seed} — saving to {seed_results_dir}")
    print(f"{'═'*70}")

    for dataset in DATASETS:
        for target_type in TARGET_TYPES:
            for split_name in ALL_SPLITS:
                if (seed, dataset, target_type, split_name) in already:
                    print(f"  skip (done): {dataset}/{split_name}/{target_type}/{seed}")
                    completed += 1
                    continue
                try:
                    row = run_single_retrain(split_name, dataset, target_type,
                                             seed, seed_results_dir)
                    all_results.append(row)
                    robust_append_csv(row_path, pd.DataFrame([row]))
                    completed += 1
                    elapsed = time.time() - total_start
                    done_now = max(1, completed - len(already))
                    rate = elapsed / done_now
                    print(f"\n  ✓ {completed}/{total_runs}  "
                          f"({elapsed/60:.1f}min elapsed, "
                          f"~{rate*(total_runs-completed)/60:.1f}min remaining)")
                except Exception as e:
                    failed += 1
                    print(f"\n  ✗ FAILED ({failed}): seed={seed} "
                          f"{dataset}/{split_name}/{target_type}: {e}")
                    import traceback; traceback.print_exc()
                    continue

print(f"\n\n{'='*70}")
print(f"  FINISHED: {completed}/{total_runs} completed, {failed} failed")
print(f"  Total time: {(time.time()-total_start)/60:.1f} minutes")
print(f"{'='*70}")


# %% [markdown]
# ## Cross-seed summary + the three-arm comparison

# %%
if row_path.exists():
    df = pd.read_csv(row_path)

    binary_df = df[df["target"] == "binary"]
    cont_df   = df[df["target"] == "continuous"]

    print("\n" + "=" * 70)
    print("  SPARSE KAN v3 — Binary Test AUC (mean ± std across seeds)")
    print("=" * 70)
    if len(binary_df):
        print(binary_df.groupby(["dataset", "split"])["test_auc"]
                       .agg(["mean", "std"]).to_string())

    for col, lbl in [("test_r2", "Test R²"),
                     ("test_mse", "Test MSE (pct², PRIMARY METRIC)"),
                     ("test_derived_auc", "Derived AUC")]:
        if col in cont_df.columns and len(cont_df):
            print(f"\n  SPARSE KAN v3 — {lbl} (mean ± std across seeds)")
            print("  " + "-" * 66)
            print(cont_df.groupby(["dataset", "split"])[col]
                         .agg(["mean", "std"]).to_string())

    # ── Seed variance, directly comparable to v1 (0.052) and v2 (0.141) ──
    print("\n" + "=" * 70)
    print("  SEED VARIANCE (compare: v1 binary 0.052, v2 binary 0.141)")
    print("=" * 70)
    for tt, col, lbl in [("binary", "test_auc", "AUC"),
                         ("continuous", "test_derived_auc", "Derived AUC")]:
        sub = df[df["target"] == tt]
        if col in sub.columns and len(sub):
            per_cfg = sub.groupby(["dataset", "split"])[col].std()
            print(f"  {tt.upper()} ({lbl}): mean seed std = {per_cfg.mean():.4f}, "
                  f"max = {per_cfg.max():.4f}")

    # ── THE TABLE THAT MATTERS: like-for-like retrain vals ──
    print("\n" + "=" * 90)
    print("  THREE-ARM COMPARISON — final-retrain validation metric (like-for-like)")
    print("  v1: pre-fix, 70 trials | v2: fixed, 0 trials | v3: fixed, 16 trials")
    print("=" * 90)

    comp = df[["seed", "dataset", "split", "target",
               "v1_val_retrain", "v2_val_retrain", "v3_val_retrain",
               "v3_search_max"]].copy()
    comp["v2_minus_v1"] = comp["v2_val_retrain"] - comp["v1_val_retrain"]
    comp["v3_minus_v1"] = comp["v3_val_retrain"] - comp["v1_val_retrain"]
    comp["v3_minus_v2"] = comp["v3_val_retrain"] - comp["v2_val_retrain"]
    comp["searchmax_bias"] = comp["v3_search_max"] - comp["v3_val_retrain"]

    print(comp.to_string(index=False))

    print(f"\n  MEANS (n={comp['v3_minus_v1'].notna().sum()}):")
    print(f"    v2 - v1 : {comp['v2_minus_v1'].mean():+.4f}   "
          f"(v2 better in {(comp['v2_minus_v1'] > 0).sum()} of "
          f"{comp['v2_minus_v1'].notna().sum()})")
    print(f"    v3 - v1 : {comp['v3_minus_v1'].mean():+.4f}   "
          f"(v3 better in {(comp['v3_minus_v1'] > 0).sum()} of "
          f"{comp['v3_minus_v1'].notna().sum()})")
    print(f"    v3 - v2 : {comp['v3_minus_v2'].mean():+.4f}   "
          f"(v3 better in {(comp['v3_minus_v2'] > 0).sum()} of "
          f"{comp['v3_minus_v2'].notna().sum()})")
    print(f"\n    Optuna search-max bias (search_max - achieved retrain val):")
    print(f"      mean {comp['searchmax_bias'].mean():+.4f}, "
          f"positive in {(comp['searchmax_bias'] > 0).sum()} of "
          f"{comp['searchmax_bias'].notna().sum()}")
    print(f"      ^ this is the artefact that made the raw probe summary "
          f"look like a v3 win.")

    comp.to_csv(RESULTS_DIR / "three_arm_comparison.csv", index=False)
    print(f"\n  Saved: {RESULTS_DIR / 'three_arm_comparison.csv'}")


# %% [markdown]
# ## Backtests (seed-averaged signal)

# %%
def load_averaged_predictions(model_name, split_name, target_type, part,
                              seeds, results_dir):
    signals, returns = [], None
    for seed in seeds:
        loaded = load_predictions(
            model_name=model_name, split_name=split_name,
            target_type=target_type, part=part,
            results_dir=results_dir / f"seed_{seed}",
        )
        preds = loaded["predictions"]
        if returns is None:
            returns = preds["daily_return"].values
        signals.append(preds["y_prob"].values if target_type == "binary"
                       else preds["y_pred"].values)
    return returns, np.mean(np.stack(signals, axis=0), axis=0)


def _json_safe(obj):
    if isinstance(obj, dict):  return {k: _json_safe(v) for k, v in obj.items()}
    if isinstance(obj, pd.DataFrame): return obj.reset_index().to_dict(orient="records")
    if isinstance(obj, (np.floating, np.integer)): return obj.item()
    if isinstance(obj, np.ndarray): return obj.tolist()
    return obj


print("\n" + "=" * 70)
print("  SPARSE KAN v3 — BACKTESTS (signal = mean prediction across seeds)")
print("=" * 70)

backtest_dir = RESULTS_DIR / "backtests"
backtest_dir.mkdir(parents=True, exist_ok=True)
bt_rows, bt_records = [], {}

for dataset in DATASETS:
    for split_name in ALL_SPLITS:
        model_name = f"sparse_kan_{dataset}"
        try:
            vr, vs = load_averaged_predictions(model_name, split_name, "binary",
                                               "val", SEEDS, RESULTS_DIR)
            tr, ts = load_averaged_predictions(model_name, split_name, "binary",
                                               "test", SEEDS, RESULTS_DIR)
            bt_bin = run_full_backtest(
                val_returns=vr, val_signal=vs, test_returns=tr, test_signal=ts,
                go_cash_when="above",
                model_name=f"{model_name}_v3 (binary)", split_name=split_name)

            vr, vs = load_averaged_predictions(model_name, split_name, "continuous",
                                               "val", SEEDS, RESULTS_DIR)
            tr, ts = load_averaged_predictions(model_name, split_name, "continuous",
                                               "test", SEEDS, RESULTS_DIR)
            bt_con = run_full_backtest(
                val_returns=vr, val_signal=vs, test_returns=tr, test_signal=ts,
                go_cash_when="below",
                model_name=f"{model_name}_v3 (continuous)", split_name=split_name)
        except Exception as e:
            print(f"  ⚠ backtest skipped for {dataset}/{split_name}: {e}")
            continue

        bt_records[f"{dataset}/{split_name}"] = {"binary": bt_bin,
                                                 "continuous": bt_con}
        for tt, bt in [("binary", bt_bin), ("continuous", bt_con)]:
            for sname, skey in [("simple", "simple"), ("risk_scaled", "risk_scaled")]:
                r = bt[skey]
                bt_rows.append({
                    "dataset": dataset, "split": split_name, "target_type": tt,
                    "strategy": sname, "sharpe": r["sharpe"], "sortino": r["sortino"],
                    "annual_return": r["annual_return"],
                    "max_drawdown": r["max_drawdown"],
                    "avg_exposure": r["avg_exposure"],
                    "annual_turnover": r["annual_turnover"],
                    "buy_hold_sharpe": r["buy_hold_sharpe"],
                    "buy_hold_sortino": r["buy_hold_sortino"],
                })

if bt_rows:
    pd.DataFrame(bt_rows).to_csv(backtest_dir / "backtest_summary.csv", index=False)
    with open(backtest_dir / "backtest_full_results.json", "w") as f:
        json.dump(_json_safe(bt_records), f, indent=2, default=str)
    print(f"\n  Backtest summary: {backtest_dir / 'backtest_summary.csv'}")
    print(f"  Full results:     {backtest_dir / 'backtest_full_results.json'}")

    # NOTE FOR WRITE-UP: Sortino is degenerate when avg_exposure is tiny
    # (no down days in the invested subset). Suppress or flag it there.
    degen = pd.DataFrame(bt_rows).query("avg_exposure < 0.05")
    if len(degen):
        print(f"\n  ⚠ {len(degen)} strategy cells have avg_exposure < 5% -- "
              f"their Sortino figures are degenerate, do not quote them.")

print("\nDone. Arms on disk: sparse_kan/ (v1), sparse_kan_v2/ (v2), "
      "sparse_kan_v3/ (v3).")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4
DRIVE HEALTH CHECK
  ✓ Drive write/read/delete OK

SANITY CHECK: init fix + eps=1e-5 / affine=False
  ✓ eps=1e-5, affine=False on both bn1/bn2
  ✓ init fix on base_weight: ratio = 0.968 (expect ~0.83)
  ✓ init fix on spline_scaler: ratio = 0.731 (expect ~0.83)
  ✓ All masked parameters are exactly zero and finite
  ✓ forward pass OK in train mode

SANITY CHECK PASSED
Device: Tesla T4 (CUDA)

Taxonomies:
  agg_full_moments: 1699 features, 331 subthemes, 13 themes
  agg_means: 574 features, 128 subthemes, 13 themes

PRE-FLIGHT: auditing v3 probe studies
  missing studies:        4
    sparse_kan_agg_full_moments_continuous_Split_C_v3probe_seed42
    sparse_kan_agg_means_binary_Split

Traceback (most recent call last):
  File "/tmp/ipykernel_1314/2952686017.py", line 509, in <cell line: 0>
    row = run_single_retrain(split_name, dataset, target_type,
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1314/2952686017.py", line 344, in run_single_retrain
    best_params, v3_search_max, n_trials = load_v3_best_params(
                                           ^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1314/2952686017.py", line 283, in load_v3_best_params
    raise FileNotFoundError(f"v3 probe study not found at {p}")
FileNotFoundError: v3 probe study not found at /content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/sparse_kan_v3_probe/optuna/sparse_kan_agg_full_moments_continuous_Split_C_seed42_v3probe.db


  v3 params (16 trials, search max=0.1715): {'lr': 0.0004066563313514797, 'weight_decay': 2.4586032763280077e-06, 'batch_size': 64, 'reg_weight': 9.565499215943809e-06}
  Epoch    1 | Train Huber 1.3433 | Val Huber 1.9571  MSE 4.1471  R² -2.6905 | LR 4.1e-04
    [act @ ep 1]  L1 pre-BN std= 0.477 post-BN std=0.958 clamp=0.273%  |  L2 pre-BN std= 0.362 post-BN std=0.932 clamp=0.049%
  Epoch   20 | Train Huber 0.3337 | Val Huber 0.7052  MSE 1.4335  R² -0.2757 | LR 2.0e-04
    [act @ ep 20]  L1 pre-BN std= 0.425 post-BN std=1.012 clamp=0.398%  |  L2 pre-BN std= 0.352 post-BN std=0.981 clamp=0.101%
  Early stop at epoch 31. Best val R²: -0.0792 at epoch 11
  Training complete in 27.7s
  ✓ All masked parameters are exactly zero and finite

  Results (v3, seed=42, huber_delta=2.4998):
    Train R2: 0.5157
    Val R2:   -0.0792
    Test R2:  -0.3927  (MSE=0.6902)
    Derived AUC: 0.4734
    ── retrain val:  v1=0.0087   v2=0.2057   v3=-0.0792
    ── (v3 Optuna search max was 0.1715 -- NOT comp

Traceback (most recent call last):
  File "/tmp/ipykernel_1314/2952686017.py", line 509, in <cell line: 0>
    row = run_single_retrain(split_name, dataset, target_type,
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1314/2952686017.py", line 344, in run_single_retrain
    best_params, v3_search_max, n_trials = load_v3_best_params(
                                           ^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1314/2952686017.py", line 283, in load_v3_best_params
    raise FileNotFoundError(f"v3 probe study not found at {p}")
FileNotFoundError: v3 probe study not found at /content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/sparse_kan_v3_probe/optuna/sparse_kan_agg_means_binary_Split_B_seed42_v3probe.db
Traceback (most recent call last):
  File "/tmp/ipykernel_1314/2952686017.py", line 509, in <cell line: 0>
    row = run_single_retrain(split_name, dataset, target_type,
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


  ✗ FAILED (3): seed=42 agg_means/Split_C/binary: v3 probe study not found at /content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/sparse_kan_v3_probe/optuna/sparse_kan_agg_means_binary_Split_C_seed42_v3probe.db

────────────────────────────────────────────────────────────
  seed=42 / agg_means / Split_D / binary
────────────────────────────────────────────────────────────
  v3 params (16 trials, search max=0.7123): {'lr': 0.00011715937392307068, 'weight_decay': 0.004337920697490943, 'batch_size': 128, 'reg_weight': 1.203017887115466e-05}
  Epoch    1 | Train loss 1.1292 | Val loss 1.4019  AUC 0.5227 | LR 1.2e-04
    [act @ ep 1]  L1 pre-BN std= 0.449 post-BN std=0.798 clamp=0.144%  |  L2 pre-BN std= 0.257 post-BN std=0.647 clamp=0.015%
  Epoch   20 | Train loss 0.9519 | Val loss 1.2860  AUC 0.6451 | LR 1.2e-04
    [act @ ep 20]  L1 pre-BN std= 0.443 post-BN std=1.013 clamp=0.362%  |  L2 pre-BN std= 0.327 post-BN std=1.004 clamp=0.143%
  Epoch   40 | Train loss 0.8253 | V

Traceback (most recent call last):
  File "/tmp/ipykernel_1314/2952686017.py", line 509, in <cell line: 0>
    row = run_single_retrain(split_name, dataset, target_type,
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1314/2952686017.py", line 344, in run_single_retrain
    best_params, v3_search_max, n_trials = load_v3_best_params(
                                           ^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1314/2952686017.py", line 283, in load_v3_best_params
    raise FileNotFoundError(f"v3 probe study not found at {p}")
FileNotFoundError: v3 probe study not found at /content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/sparse_kan_v3_probe/optuna/sparse_kan_agg_means_continuous_Split_C_seed42_v3probe.db


  v3 params (16 trials, search max=0.1753): {'lr': 0.008681223927448443, 'weight_decay': 7.821846444899164e-05, 'batch_size': 256, 'reg_weight': 1.1104839169071245e-06}
  Epoch    1 | Train Huber 1.0795 | Val Huber 0.9781  MSE 1.9871  R² -0.7683 | LR 8.7e-03
    [act @ ep 1]  L1 pre-BN std= 0.476 post-BN std=0.649 clamp=0.042%  |  L2 pre-BN std= 0.282 post-BN std=0.460 clamp=0.000%
  Epoch   20 | Train Huber 0.1154 | Val Huber 1.4346  MSE 2.9659  R² -1.6393 | LR 2.2e-03
    [act @ ep 20]  L1 pre-BN std= 0.470 post-BN std=1.005 clamp=0.279%  |  L2 pre-BN std= 0.483 post-BN std=0.999 clamp=0.116%
  Early stop at epoch 22. Best val R²: 0.1478 at epoch 2
  Training complete in 3.5s
  ✓ All masked parameters are exactly zero and finite

  Results (v3, seed=42, huber_delta=2.4998):
    Train R2: 0.4020
    Val R2:   0.1478
    Test R2:  -0.8802  (MSE=0.9317)
    Derived AUC: 0.6290
    ── retrain val:  v1=0.0154   v2=-0.2692   v3=0.1478
    ── (v3 Optuna search max was 0.1753 -- NOT comparab

In [3]:
# %% [markdown]
# ## Patch: re-run the 5 orphaned probe configs under the consistent
# naming convention, then retrain just those 5.
#
# These 5 configs were the original hand-picked probes (Part 1 of the
# v3 probe notebook), which saved their Optuna .db files WITHOUT a seed
# suffix (f"{model_name}_{target_type}_{split_name}_v3probe.db"), while
# every other config uses f"..._{split_name}_seed{seed}_v3probe.db".
# The pre-flight audit above correctly flagged these as "missing" under
# the consistent naming -- this patch re-runs their SEARCH under that
# same consistent naming, then retrains them, so they end up in
# three_arm_comparison.csv on the same footing as the other 43.

# %%
ORPHANED_CONFIGS = [
    {"dataset": "agg_full_moments", "target": "continuous", "split": "Split_C", "seed": 42},
    {"dataset": "agg_means",        "target": "binary",     "split": "Split_B", "seed": 42},
    {"dataset": "agg_means",        "target": "binary",     "split": "Split_C", "seed": 42},
    {"dataset": "agg_means",        "target": "continuous", "split": "Split_C", "seed": 42},
    # The under-searched one (4/16 trials) -- top up rather than re-run
    # from scratch; optuna.create_study(load_if_exists=True) will resume it.
    {"dataset": "agg_full_moments", "target": "binary",     "split": "Split_B", "seed": 42},
]

N_TRIALS_PROBE = 15  # matches the original probe notebook's per-config budget

def make_model_factory_with_l1_patch(feature_cols, taxonomy_df, huber_delta, target_type):
    def factory(trial):
        lr           = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
        batch_size   = trial.suggest_categorical("batch_size", [64, 128, 256])
        reg_weight   = trial.suggest_float("reg_weight", 1e-7, 1e-3, log=True)

        model = SparseKAN.from_taxonomy(
            taxonomy_df, feature_cols,
            grid_size=GRID_SIZE, spline_order=SPLINE_ORDER, grid_range=GRID_RANGE,
        )
        train_kwargs = {
            "lr": lr, "weight_decay": weight_decay,
            "reg_fn": sparse_kan_edge_l1, "reg_weight": reg_weight,
            "n_epochs": 300, "patience": 20,
        }
        if target_type == "continuous":
            train_kwargs["huber_delta"] = huber_delta
        return model, train_kwargs
    return factory


def load_v1_best_params_for_enqueue(model_name, target_type, split_name, seed):
    """Same helper as the original v3 probe notebook -- v1's with-L1 study,
    used as the enqueue_trial(v1_best) starting point."""
    study_dir = V1_RESULTS_DIR / f"seed_{seed}" / "optuna"
    path_l1 = study_dir / f"{model_name}_{target_type}_{split_name}_with_L1_seed{seed}.db"
    if not path_l1.exists():
        raise FileNotFoundError(f"v1 with-L1 study not found at {path_l1}")
    study = optuna.load_study(
        study_name=f"{model_name}_{target_type}_{split_name}_with_L1_seed{seed}",
        storage=f"sqlite:///{path_l1}",
    )
    return study.best_params


print("=" * 90)
print(f"PATCHING {len(ORPHANED_CONFIGS)} ORPHANED PROBE CONFIGS")
print("(re-running/topping-up their Optuna search under the CONSISTENT")
print(" naming convention: ..._{split}_seed{seed}_v3probe.db)")
print("=" * 90)

for cfg in ORPHANED_CONFIGS:
    dataset, target_type, split_name, seed = (
        cfg["dataset"], cfg["target"], cfg["split"], cfg["seed"]
    )
    model_name = f"sparse_kan_{dataset}"

    print(f"\n{'─'*70}")
    print(f"  PATCHING {model_name} / {target_type} / {split_name} / seed{seed}")
    print(f"{'─'*70}")

    set_seed(seed)

    data         = load_split(split_name, dataset, SPLITS_DIR)
    feature_cols = data["feature_cols"]
    taxonomy_df  = taxonomy_dfs[dataset]

    n_pos      = data["y_train"].sum()
    n_neg      = len(data["y_train"]) - n_pos
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32)
    huber_delta = get_huber_delta(split_name) if target_type == "continuous" else None

    # Use the SAME consistent path pattern the retrain script's
    # v3_study_path()/v3_study_name() already expect -- no special-casing
    # needed anywhere else once this is written correctly.
    study_path = v3_study_path(model_name, target_type, split_name, seed)
    study_path.parent.mkdir(parents=True, exist_ok=True)

    study = optuna.create_study(
        study_name=v3_study_name(model_name, target_type, split_name, seed),
        storage=f"sqlite:///{study_path}",
        direction="maximize",
        load_if_exists=True,   # resumes the 4/16 case cleanly
        sampler=optuna.samplers.TPESampler(seed=seed),
    )

    factory = make_model_factory_with_l1_patch(feature_cols, taxonomy_df, huber_delta, target_type)

    def objective(trial):
        model, train_kwargs = factory(trial)
        batch_size = trial.params["batch_size"]
        loaders = get_dataloaders(split_name, dataset, SPLITS_DIR,
                                  target_type=target_type, batch_size=batch_size)
        result = train_model(
            model=model, train_loader=loaders["train"], val_loader=loaders["val"],
            device=device, target_type=target_type,
            pos_weight=pos_weight if target_type == "binary" else None,
            verbose=False, **train_kwargs,
        )
        return result["best_val_metric"]

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    existing = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)

    if existing == 0:
        v1_best_params = load_v1_best_params_for_enqueue(model_name, target_type, split_name, seed)
        study.enqueue_trial(v1_best_params)
        print(f"  Enqueued v1 best params as trial 0")

    target_total = N_TRIALS_PROBE + 1  # +1 for the enqueued trial
    remaining = max(0, target_total - existing)

    if remaining > 0:
        print(f"  Running {remaining} trials ({existing} already complete, target {target_total})")
        study.optimize(objective, n_trials=remaining, show_progress_bar=True)
    else:
        print(f"  Already has {existing}/{target_total} trials — skipping")

    print(f"  Done: {sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)} "
          f"trials, best={study.best_value:.4f}")

print("\n" + "=" * 90)
print("PATCH COMPLETE — all 5 configs now have Optuna studies under the "
      "consistent naming convention")
print("=" * 90)

PATCHING 5 ORPHANED PROBE CONFIGS
(re-running/topping-up their Optuna search under the CONSISTENT
 naming convention: ..._{split}_seed{seed}_v3probe.db)

──────────────────────────────────────────────────────────────────────
  PATCHING sparse_kan_agg_full_moments / continuous / Split_C / seed42
──────────────────────────────────────────────────────────────────────
  Enqueued v1 best params as trial 0
  Running 16 trials (0 already complete, target 16)


  0%|          | 0/16 [00:00<?, ?it/s]

  Done: 16 trials, best=0.1863

──────────────────────────────────────────────────────────────────────
  PATCHING sparse_kan_agg_means / binary / Split_B / seed42
──────────────────────────────────────────────────────────────────────
  Enqueued v1 best params as trial 0
  Running 16 trials (0 already complete, target 16)


  0%|          | 0/16 [00:00<?, ?it/s]

  Done: 16 trials, best=0.7620

──────────────────────────────────────────────────────────────────────
  PATCHING sparse_kan_agg_means / binary / Split_C / seed42
──────────────────────────────────────────────────────────────────────
  Enqueued v1 best params as trial 0
  Running 16 trials (0 already complete, target 16)


...

  Done: 16 trials, best=0.7518

──────────────────────────────────────────────────────────────────────
  PATCHING sparse_kan_agg_means / continuous / Split_C / seed42
──────────────────────────────────────────────────────────────────────
  Enqueued v1 best params as trial 0
  Running 16 trials (0 already complete, target 16)


  0%|          | 0/16 [00:00<?, ?it/s]

  Done: 16 trials, best=0.2219

──────────────────────────────────────────────────────────────────────
  PATCHING sparse_kan_agg_full_moments / binary / Split_B / seed42
──────────────────────────────────────────────────────────────────────
  Running 12 trials (4 already complete, target 16)


  0%|          | 0/12 [00:00<?, ?it/s]

  Done: 16 trials, best=0.8105

PATCH COMPLETE — all 5 configs now have Optuna studies under the consistent naming convention


In [4]:
# %% [markdown]
# ## Retrain the 5 patched configs and merge into the existing results

# %%
print("=" * 90)
print(f"RETRAINING {len(ORPHANED_CONFIGS)} PATCHED CONFIGS")
print("=" * 90)

patched_rows = []
for cfg in ORPHANED_CONFIGS:
    dataset, target_type, split_name, seed = (
        cfg["dataset"], cfg["target"], cfg["split"], cfg["seed"]
    )
    seed_results_dir = RESULTS_DIR / f"seed_{seed}"
    try:
        row = run_single_retrain(split_name, dataset, target_type, seed, seed_results_dir)
        patched_rows.append(row)
        robust_append_csv(row_path, pd.DataFrame([row]))
        print(f"  ✓ patched: {dataset}/{split_name}/{target_type}/seed{seed}")
    except Exception as e:
        print(f"  ✗ patch retrain failed for {cfg}: {e}")
        import traceback; traceback.print_exc()

print(f"\n{len(patched_rows)}/{len(ORPHANED_CONFIGS)} patched configs retrained "
      f"and appended to {row_path}")

# ── Rebuild the three-arm comparison and cross-seed summaries now that
# all 48 configs are present. This just re-runs the same aggregation
# cell from earlier against the now-complete all_seeds_raw.csv. ──

df = pd.read_csv(row_path)
print(f"\nTotal rows in all_seeds_raw.csv: {len(df)} (expect 48)")

comp = df[["seed", "dataset", "split", "target",
           "v1_val_retrain", "v2_val_retrain", "v3_val_retrain",
           "v3_search_max"]].copy()
comp["v2_minus_v1"] = comp["v2_val_retrain"] - comp["v1_val_retrain"]
comp["v3_minus_v1"] = comp["v3_val_retrain"] - comp["v1_val_retrain"]
comp["v3_minus_v2"] = comp["v3_val_retrain"] - comp["v2_val_retrain"]
comp["searchmax_bias"] = comp["v3_search_max"] - comp["v3_val_retrain"]

print("\n" + "=" * 90)
print("  THREE-ARM COMPARISON (UPDATED, all 48 configs)")
print("=" * 90)
print(comp.to_string(index=False))
print(f"\n  MEANS (n={comp['v3_minus_v1'].notna().sum()}):")
print(f"    v2 - v1 : {comp['v2_minus_v1'].mean():+.4f}  "
      f"(v2 better in {(comp['v2_minus_v1'] > 0).sum()} of {comp['v2_minus_v1'].notna().sum()})")
print(f"    v3 - v1 : {comp['v3_minus_v1'].mean():+.4f}  "
      f"(v3 better in {(comp['v3_minus_v1'] > 0).sum()} of {comp['v3_minus_v1'].notna().sum()})")
print(f"    v3 - v2 : {comp['v3_minus_v2'].mean():+.4f}  "
      f"(v3 better in {(comp['v3_minus_v2'] > 0).sum()} of {comp['v3_minus_v2'].notna().sum()})")

comp.to_csv(RESULTS_DIR / "three_arm_comparison.csv", index=False)
print(f"\n  Updated: {RESULTS_DIR / 'three_arm_comparison.csv'}")

RETRAINING 5 PATCHED CONFIGS

────────────────────────────────────────────────────────────
  seed=42 / agg_full_moments / Split_C / continuous
────────────────────────────────────────────────────────────
  v3 params (16 trials, search max=0.1863): {'lr': 0.0012399967836846098, 'weight_decay': 5.4880470007660465e-06, 'batch_size': 64, 'reg_weight': 0.0003795853142670637}
  Epoch    1 | Train Huber 0.9081 | Val Huber 1.1579  MSE 3.5401  R² -0.2537 | LR 1.2e-03
    [act @ ep 1]  L1 pre-BN std= 0.417 post-BN std=0.949 clamp=0.277%  |  L2 pre-BN std= 0.344 post-BN std=0.941 clamp=0.169%
  Epoch   20 | Train Huber 0.1987 | Val Huber 1.5470  MSE 4.0390  R² -0.4304 | LR 3.1e-04
    [act @ ep 20]  L1 pre-BN std= 0.222 post-BN std=0.983 clamp=0.326%  |  L2 pre-BN std= 0.281 post-BN std=0.996 clamp=0.090%
  Early stop at epoch 23. Best val R²: 0.0527 at epoch 3
  Training complete in 18.7s
  ✓ All masked parameters are exactly zero and finite

  Results (v3, seed=42, huber_delta=2.4230):
    Trai